# MethylGPT Quickstart

Get started with MethylGPT in 5 minutes. This notebook demonstrates:

1. Installing MethylGPT
2. Loading a pretrained model
3. Extracting embeddings from sample data
4. Visualizing results with UMAP

**Works on Google Colab (GPU recommended) and locally.**

In [ ]:
# Cell 1: Environment setup
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q methylgpt[tutorials]
    !pip install -q gdown
    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: No GPU. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Cell 2: Imports
import os
import json
import warnings
from pathlib import Path

import numpy as np
import torch

from methylgpt import MethylGPTModel, MethylVocab, create_dataloader
from methylgpt.inference import extract_embeddings

warnings.filterwarnings("ignore", message=".*IProgress.*")
warnings.filterwarnings("ignore", message=".*flash_attn.*")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"MethylGPT ready!")

## Download Model & Data

Download the **methylGPT-base** model (smallest, 3M params) and a sample Parquet file.

If running locally, download from:
- **Model**: [Google Drive](https://drive.google.com/drive/folders/1kWdmkkVQpU17uzUC6-wpNR_4UEdxGx6k)
- **Probe IDs**: [Dropbox](https://www.dropbox.com/scl/fi/2n6bx7j8v0aon0kwfsghp/probe_ids_type3.csv?rlkey=ly133xlce1xxjiku6tiski6qq&st=pig4e41h&dl=0)
- **Data**: [Dropbox](https://www.dropbox.com/scl/fi/bbs6sxlkpbx11rhyvdfto/processed_type3_parquet_shuffled.tar.gz?rlkey=s73utmumq6xldmv3y6kh9bz75&st=8pslwy2a&dl=0)

In [ ]:
# Cell 3: Configure paths
# === UPDATE THESE PATHS for your local setup ===
MODEL_DIR = "pretrained_models/methylgpt-base"  # Directory with args.json + model .pt
CPG_LIST_FILE = "data/probe_ids_type3.csv"      # CpG probe IDs
PARQUET_DIR = "data/processed_type3_parquet_shuffled"  # Parquet data files

# On Colab: download model, probe IDs, and sample data
if IN_COLAB:
    import gdown, subprocess
    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs("data", exist_ok=True)

    # 1. Download tiny pretrained model
    if not list(Path(MODEL_DIR).glob("*.pt")):
        print("Downloading methylgpt-base model...")
        gdown.download_folder(
            "https://drive.google.com/drive/folders/1kWdmkkVQpU17uzUC6-wpNR_4UEdxGx6k",
            output=MODEL_DIR, quiet=True
        )
        print(f"Model downloaded to {MODEL_DIR}/")
    else:
        print(f"Model already exists in {MODEL_DIR}/")

    # 2. Download probe_ids_type3.csv
    if not os.path.exists(CPG_LIST_FILE):
        print("Downloading probe_ids_type3.csv...")
        subprocess.run([
            "wget", "-q", "-O", CPG_LIST_FILE,
            "https://www.dropbox.com/scl/fi/2n6bx7j8v0aon0kwfsghp/probe_ids_type3.csv?rlkey=ly133xlce1xxjiku6tiski6qq&st=pig4e41h&dl=1"
        ], check=True)
        print(f"Probe IDs saved to {CPG_LIST_FILE}")
    else:
        print(f"Probe IDs already exist at {CPG_LIST_FILE}")

    # 3. Download sample parquet data
    if not os.path.exists(PARQUET_DIR):
        print("Downloading sample parquet data (~2 GB)...")
        subprocess.run([
            "wget", "-q", "--show-progress", "-O", "data/parquet_data.tar.gz",
            "https://www.dropbox.com/scl/fi/bbs6sxlkpbx11rhyvdfto/processed_type3_parquet_shuffled.tar.gz?rlkey=s73utmumq6xldmv3y6kh9bz75&st=8pslwy2a&dl=1"
        ], check=True)
        subprocess.run(["tar", "-xzf", "data/parquet_data.tar.gz", "-C", "data/"], check=True)
        os.remove("data/parquet_data.tar.gz")
        print(f"Parquet data extracted to {PARQUET_DIR}")
    else:
        print(f"Parquet data already exists at {PARQUET_DIR}")


In [ ]:
# Cell 4: Load model
# Load config
with open(Path(MODEL_DIR) / "args.json", "r") as f:
    config = json.load(f)

# Find model checkpoint
model_files = list(Path(MODEL_DIR).glob("*.pt"))
assert model_files, f"No .pt files in {MODEL_DIR}. Download model first."
model_file = str(model_files[0])

config["load_model"] = True
config["pretrained_file"] = model_file
config["mask_ratio"] = 0  # No masking for inference

# Load vocabulary
vocab = MethylVocab(
    probe_id_dir=CPG_LIST_FILE,
    pad_token="<pad>",
    special_tokens=["<pad>", "<cls>", "<eoc>"],
    save_dir=None,
)

# Load model
model = MethylGPTModel.from_pretrained(config, vocab)
model.eval()
model.to(device)
if device.type == "cuda":
    model.half()

print(f"Loaded methylGPT-base: {config['layer_size']}-dim, {config['nlayers']} layers")

In [ ]:
# Cell 5: Extract embeddings
parquet_files = sorted([
    os.path.join(PARQUET_DIR, f) for f in os.listdir(PARQUET_DIR)
    if f.endswith(".parquet")
])
print(f"Found {len(parquet_files)} data files")

# Use first file for demo
data_loader = create_dataloader([parquet_files[0]], batch_size=32)

# Extract embeddings (limit to 50 batches for demo speed)
embeddings, sample_ids = extract_embeddings(
    model, data_loader, device=str(device), max_batches=50
)
print(f"Extracted {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]}")

In [ ]:
# Cell 6: Visualize with UMAP
import umap
import matplotlib.pyplot as plt
from aquarel import load_theme

theme = (
    load_theme("scientific")
    .set_grid(draw=False)
    .set_font(size=15)
    .set_ticks(direction="out")
    .set_axis_labels(pad=10)
)
theme.apply()

# Reduce to 2D
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
coords = reducer.fit_transform(embeddings)

# Plot with outline layer
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(coords[:, 0], coords[:, 1], s=12, c="black", alpha=1, zorder=1)
ax.scatter(coords[:, 0], coords[:, 1], s=8, alpha=0.6, c="steelblue", zorder=2)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title(f"MethylGPT Embeddings ({embeddings.shape[0]} samples)")

theme.apply_transforms()

plt.savefig("quickstart_umap.pdf", bbox_inches="tight")
plt.savefig("quickstart_umap.png", dpi=600, bbox_inches="tight")
plt.show()

print(f"Done! Each sample is represented as a {embeddings.shape[1]}-dimensional vector.")

## Next Steps

| Tutorial | Description | Link |
|----------|-------------|------|
| **Embedding Extraction** | Full embedding pipeline with metadata | [tutorials/get_embeddings/](../get_embeddings/) |
| **Embedding Analysis** | UMAP by tissue/age/sex, clustering | [tutorials/embedding_analysis/](../embedding_analysis/) |
| **Age Prediction** | Finetune for biological age | [tutorials/finetuning_age_prediction/](../finetuning_age_prediction/) |
| **Disease Prediction** | Survival analysis with embeddings | [tutorials/disease_prediction/](../disease_prediction/) |
| **Imputation** | Recover missing CpG values | [tutorials/imputation/](../imputation/) |

See the full [documentation](../../docs/) for the inference guide, API reference, and troubleshooting.